In [1]:
from coffea.nanoevents import NanoAODSchema
from coffea.dataset_tools import apply_to_fileset, max_chunks, max_files, preprocess

import dask

from analysis_tools.processors.test_processor import TestProcessor


/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/fcc.py:5: FutureWarning: In version 2025.1.0 (target date: 2024-12-31 11:59:59-06:00), this will be an error.
To raise these warnings as errors (and get stack traces to find out where they're called), run
    import warnings
    warnings.filterwarnings("error", module="coffea.*")
after the first `import coffea` or use `@pytest.mark.filterwarnings("error:::coffea.*")` in pytest.
Issue: coffea.nanoevents.methods.vector will be removed and replaced with scikit-hep vector. Nanoevents schemas internal to coffea will be migrated. Otherwise please consider using that package!.
  from coffea.nanoevents.methods import vector


In [2]:
import gzip
import json
import os

# Define the base directory where the preprocessed files are stored
base_dir = "/home/cms-jovyan/dwg_analysis/tools/preprocessing/preprocessed"
signal_sample = "2023_SlepSnu_MN1_270_100000_preprocessed_available.json.gz"
background_sample = "2023_ttbar_100000_preprocessed_available.json.gz"
signal_file_path = os.path.join(base_dir, signal_sample)
background_file_path = os.path.join(base_dir, background_sample)
#print(preprocessed)

with gzip.open(signal_file_path, "rt") as file:
    signal_preprocessed_available = json.load(file)
with gzip.open(background_file_path, "rt") as file:
    background_preprocessed_available = json.load(file)

In [3]:
signal_test_preprocessed_files = max_files(signal_preprocessed_available, 3)
signal_test_preprocessed = max_chunks(signal_test_preprocessed_files, 3)

### SWITCH HERE ###

signal_reduced_computation = True

###################

# signal

if signal_reduced_computation:
    small_tg, small_rep = apply_to_fileset(
        data_manipulation=TestProcessor(),
        fileset=signal_test_preprocessed,
        schemaclass=NanoAODSchema,
        uproot_options={"allow_read_errors_with_report": (OSError, KeyError)},
    )
    signal_computed, rep = dask.compute(small_tg, small_rep)
    
else:
    full_tg, full_rep = apply_to_fileset(
        data_manipulation=TestProcessor(),
        fileset=signal_preprocessed_available,
        schemaclass=NanoAODSchema,
        uproot_options={"allow_read_errors_with_report": (OSError, KeyError)},
    )
    signal_computed, rep = dask.compute(full_tg, full_rep)


['genPartIdxMother', 'statusFlags', 'pdgId', 'status', 'eta', 'mass', 'phi', 'pt', 'genPartIdxMotherG', 'distinctParentIdxG', 'childrenIdxG', 'distinctChildrenIdxG', 'distinctChildrenDeepIdxG']


In [4]:
signal_results = signal_computed['/SlepSnuCascade_MN1-270_MN2-280_MC1-275_TuneCP5_13p6TeV_madgraphMLM-pythia8/Run3Summer23BPixNanoAODv12-130X_mcRun3_2023_realistic_postBPix_v6-v3/NANOAODSIM']
signal_results

{'counts': {'count_obj_test': 228295,
  'count_obj_events_test': 154274,
  'count_obj_and_events_test': (228295, 154274),
  'filter_ele_count': 47993,
  'count_gen_kin': 178121,
  'count_primary_vertex': 169297,
  'count_parent': 48716,
  'test_gen_mask': <Array [[False], [None], ..., [False, ...], []] type='10 * var * ?bool[para...'>,
  'test_vertex_mask': <Array [[False], [False], ..., [True, ...], []] type='10 * var * bool[param...'>,
  'test_ele': <ElectronArray [[{seediEtaOriX: 48, ...}], ..., []] type='10 * var * Electr...'>,
  'test_kin_mask': <Array [[True], [], [True, ...], ..., [True], []] type='303452 * var * bool'>},
 'pt_binned': {},
 'calculations': {},
 'plots': {'ele_pt_dist': Hist(Regular(1000, 5, 200, name='pt_dist'), storage=Double()) # Sum: 227142.0 (228295.0 with flow),
  'ele_pt_eta_dist': Hist(
    Regular(100, 0, 10, name='pt'),
    Regular(100, -3, 3, name='eta'),
    storage=Double()) # Sum: 59531.0 (228295.0 with flow),
  'hoe_dist_baseline': Hist(Regular(100

In [5]:
test_gen_mask = signal_results['counts']['test_gen_mask']
test_gen_mask

<Array [[False], [None], ..., [False, ...], []] type='10 * var * ?bool[para...'>

In [6]:
test_vertex_mask = signal_results['counts']['test_vertex_mask']
test_vertex_mask

<Array [[False], [False], ..., [True, ...], []] type='10 * var * bool[param...'>

In [7]:
combined_mask = test_gen_mask & test_vertex_mask
combined_mask

<Array [[False], [None], ..., [False, False], []] type='10 * var * ?bool'>

In [8]:
test_ele = signal_results['counts']['test_ele']
print(test_ele)

[[{seediEtaOriX: 48, convVeto: True, cutBased: 0, ...}], [{...}], ..., []]


In [9]:
test_kin_mask = signal_results['counts']['test_kin_mask']
test_kin_mask

<Array [[True], [], [True, ...], ..., [True], []] type='303452 * var * bool'>

```
[[True],
 [],
 [True, True],
 [True],
 [True],
 [True],
 [True, None, None],
 [True, True],
 [True],
 [],
 ...,
 [True, True],
 [],
 [None],
 [],
 [True, True],
 [True],
 [None],
 [True],
 [True, True]]
-------------------------
backend: cpu
nbytes: 384.9 kB
type: 26406 * var * ?bool
```

In [10]:
import awkward as ak
test_ele_filtered = test_ele[combined_mask]
print(test_ele_filtered)
count_ele = ak.sum(ak.num(test_ele_filtered))
print(count_ele)

[[], [None], [], [], [Electron, None], [], [], [], [], []]
3


In [11]:
test_ele_filtered


<ElectronArray [[], [None], [], [], ..., [], [], []] type='10 * var * ?Elec...'>

In [12]:
mask_1 = [True, False, True, True]
mask_2 = [True, True, True, False]
combined = mask_1 & mask_2

TypeError: unsupported operand type(s) for &: 'list' and 'list'

In [ ]:
tests = signal_results["tests"]

gen_kinematic_ele = tests['test_gen_kinematic_ele']
primary_vertex_ele = tests['test_primary_vertex_ele']
parent_ele = tests['test_parent_ele']
all_ele = tests['test_all_ele']

In [ ]:
import awkward as ak
max_val = 12
min_val = 6

print(gen_kinematic_ele[min_val:max_val].pt)
print(primary_vertex_ele[min_val:max_val].pt)
print(parent_ele[min_val:max_val].pt)
print(all_ele[min_val:max_val].pt)

In [ ]:
import matplotlib.pyplot as plt
import mplhep
#have to call this a second time to get proper scaling, a known bug
mplhep.style.use(mplhep.style.CMS)
plt.figure()
mplhep.style.use(mplhep.style.CMS)

In [ ]:
hist_obj = signal_results['plots']['ele_pt_dist']

display_bins = hist_obj[0:30]
plt.figure()
fig, ax = plt.subplots(figsize=(10, 6))
display_bins.plot1d(ax=ax, label="test")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors

In [ ]:
hist_obj_baseline = signal_results['plots']['hoe_dist_baseline']

plt.figure()

fig, ax = plt.subplots(figsize=(10, 6))
plt.yscale("log")

hist_obj_baseline[0:100].plot()


In [ ]:
hist_obj_gold = signal_results['plots']['hoe_dist_gold']

plt.figure()

fig, ax = plt.subplots(figsize=(10, 6))
plt.yscale("log")

hist_obj_gold[0:100].plot()


In [ ]:
twod_baseline = signal_results['plots']['baseline_hoe_pt_dist']

#plt.figure()

#fig, ax = plt.subplots(figsize=(10, 6))

twod_baseline.plot2d(norm=colors.LogNorm())


In [ ]:
twod_gold = signal_results['plots']['gold_hoe_pt_dist']

plt.figure()

fig, ax = plt.subplots(figsize=(10, 6))

#twod_gold[0:60, 0:30].plot(norm=colors.LogNorm())
twod_gold.plot(norm=colors.LogNorm())

In [ ]:
signal_results['plots']['ele_pt_dist'][0:100].plot1d()

In [ ]:
hist_obj = signal_results['plots']['ele_pt_eta_dist']

plt.figure()
fig, ax = plt.subplots(figsize=(10, 6))
hist_obj.plot2d(ax=ax, label="test")
#hist_obj.plot2d(ax=ax, label="test")

In [ ]:
hist_obj = signal_results['plots']['ele_pt_eta_dist']

hist_obj.plot2d_full()
#hist_obj.plot2d(ax=ax, label="test")

In [ ]:
hist_obj.axes

In [ ]:
hist_obj.axes.name[0]